# 01 · Frontier Models via Local Ollama

This notebook connects to a **local Ollama server** using the standard **OpenAI SDK** (Ollama exposes an OpenAI-compatible endpoint at `http://localhost:11434/v1`).

We'll pit two locally-hosted models against a small set of reasoning puzzles:

- `qwen3:8b`
- `deepseek-coder:6.7b`

> Make sure Ollama is running (`ollama serve`) and both models are pulled before executing the cells below.

## 1. Setup

In [1]:
from openai import OpenAI
import time

# Ollama exposes an OpenAI-compatible API — no real API key needed.
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"  # placeholder, Ollama ignores this value

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)

MODELS = ["qwen3:8b", "deepseek-coder:6.7b"]
print("Client configured for:", OLLAMA_BASE_URL)

Client configured for: http://localhost:11434/v1


## 2. Reasoning puzzles

In [2]:
PUZZLES = [
    (
        "river_crossing",
        "A farmer needs to cross a river with a fox, a chicken, and a bag "
        "of grain. The boat only fits the farmer plus one item. If left "
        "unattended, the fox eats the chicken, and the chicken eats the "
        "grain. How does the farmer get everything across safely? "
        "List the steps."
    ),
    (
        "logic_deduction",
        "Three friends, Amy, Ben, and Cara, each have a different pet: a "
        "cat, a dog, and a fish. Amy does not have the fish. The dog owner "
        "sits next to Amy. Cara owns the cat. Who owns which pet?"
    ),
    (
        "math_word_problem",
        "A shop sells apples in bags of 4 and oranges in bags of 6. "
        "A customer wants exactly 24 apples and 24 oranges using the "
        "fewest total bags. How many bags of each should they buy, and why?"
    ),
]

for name, prompt in PUZZLES:
    print(f"- {name}: {prompt[:60]}...")

- river_crossing: A farmer needs to cross a river with a fox, a chicken, and a...
- logic_deduction: Three friends, Amy, Ben, and Cara, each have a different pet...
- math_word_problem: A shop sells apples in bags of 4 and oranges in bags of 6. A...


## 3. Query helper

In [ ]:
def ask_model(model: str, prompt: str, temperature: float = 0.3) -> str:
    """Send a single-turn chat completion request to a local Ollama model."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a careful, step-by-step reasoner."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content

## 4. Run the benchmark

In [ ]:
results = {}

for model in MODELS:
    print(f"\n{'='*80}\nMODEL: {model}\n{'='*80}")
    results[model] = {}
    for name, prompt in PUZZLES:
        start = time.time()
        try:
            answer = ask_model(model, prompt)
        except Exception as e:
            answer = f"[ERROR] {e}"
        elapsed = time.time() - start
        results[model][name] = answer
        print(f"\n--- {name} ({elapsed:.1f}s) ---")
        print(answer)

## 5. Side-by-side comparison

In [ ]:
for name, _ in PUZZLES:
    print(f"\n{'#'*80}\nPUZZLE: {name}\n{'#'*80}")
    for model in MODELS:
        print(f"\n>>> {model}")
        print(results[model][name])

## Next steps

- Swap in other locally pulled models (e.g. `llama3.1:8b`, `mistral:7b`).
- Add scoring logic to automatically grade correctness.
- Try `02_adversarial_3way_chat.ipynb` for multi-agent dialogue.